# Fase 10 — Entregable 6 (bono): fusión Sentinel-2 + SAR  (Colab · 1 GPU)

**Pregunta:** ¿cuánto cambia el IoU si a la U-Net le damos también el óptico Sentinel-2 del mismo evento?

| Modo | Entrada | Canales | Para qué |
|---|---|---|---|
| `sar` | VV, VH | 2 | referencia (**re-entrenada aquí** con la misma máscara que los otros dos) |
| `optico` | B2, B3, B4, B8, B11, B12 | 6 | ¿cuánto resuelve el óptico solo? |
| `sar_optico` | VV, VH + las 6 bandas | 8 | **fusión temprana** (concatenar canales) |

**Por qué estas bandas:** visible + NIR (B8) + SWIR (B11, B12). El agua absorbe casi todo en NIR y SWIR; con ellas
están el NDWI (B3, B8) y el MNDWI (B3, B11), así que la red tiene la información de los índices clásicos.

**Rigor (igual que las ablaciones):**
- Misma U-Net (32 canales, profundidad 4, saltos), pérdida **combinada**, 60 épocas máx., paciencia 15, **3 semillas**.
- **Máscara común:** un píxel se ignora si el SAR no es finito **o** el S2 no es válido → los 3 modos se evalúan sobre los mismos píxeles. Por eso `sar` se re-entrena aquí (su IoU puede diferir un poco del 0.669 de la fase 7).
- Selección por **valid**, reporte en **test** y **Bolivia** (umbral 0.5 fijo).
- Normalización S2 calculada **solo con train** (p1/p99 + estandarización, como el SAR).
- **Δ pareado por semilla:** misma semilla = mismos lotes y aumentos → la diferencia se debe a la entrada.

**Limitación que va al informe:** el S2 de Sen1Floods11 es del **mismo día** que el S1 y los etiquetadores lo usaron para
etiquetar → el óptico tiene ventaja estructural sobre estas etiquetas. Y en un evento real con nubes, el óptico no siempre existe.

**Colab:** requiere **GPU T4**. Las 9 corridas van **en serie** en una sola GPU → **~50–60 min**.
Todo se guarda en tu Drive. **Reanudable:** si la sesión se cae, vuelve a *Ejecutar todo*. Las corridas terminadas se saltan
y la que estaba a medias continúa desde su última época (`ultimo.pt`).
Usa los módulos que ya tienes en Drive; solo escribe los **5 nuevos** (y `paralelo_6a.py` si no lo tuvieras).

## 0. Preparación

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

import sys
from pathlib import Path
RAIZ_PROYECTO = Path('/content/drive/MyDrive/rnap_p2_inundaciones')
sys.path.insert(0, str(RAIZ_PROYECTO))

!pip install -q rasterio
!nvidia-smi --query-gpu=name,memory.total --format=csv

## 1. Módulos nuevos de `src/`

In [ ]:
%%writefile /content/drive/MyDrive/rnap_p2_inundaciones/src/fusion_optica.py
"""
fusion_optica.py
Entregable 6 (bono): fusión TEMPRANA de Sentinel-2 (óptico) con Sentinel-1 (SAR).
Las bandas S2 se concatenan a VV y VH como canales extra de entrada de la MISMA U-Net.

Modos (todos evaluados sobre EXACTAMENTE los mismos píxeles):
  - "sar"        : VV, VH                      (2 canales)  -> referencia, re-entrenada aquí
  - "optico"     : B2, B3, B4, B8, B11, B12    (6 canales)  -> ¿cuánto resuelve el óptico solo?
  - "sar_optico" : VV, VH + las 6 bandas S2    (8 canales)  -> fusión

Por qué esas 6 bandas: visible (B2-B4), NIR (B8) y SWIR (B11, B12). El agua absorbe fuerte en
NIR y SWIR; con ellas se construyen el NDWI (B3, B8) y el MNDWI (B3, B11), así que la red tiene
toda la información de los índices clásicos sin que los calculemos a mano.

Máscara común: un píxel se ignora (-1) si el SAR NO es finito O si el S2 no es válido. Así la
diferencia de IoU entre modos se debe solo a la entrada, no a evaluar píxeles distintos.
Ojo: las nubes NO se enmascaran en la entrada. En Sen1Floods11 los etiquetadores ya marcaron
como -1 la mayor parte de lo nublado, pero lo que queda (bruma, bordes, sombras de nube) entra
tal cual: es parte de la limitación real del óptico.
"""
import json
from contextlib import contextmanager

import numpy as np
import torch
from torch.utils.data import DataLoader
from tqdm.auto import tqdm

from . import entrenamiento as _entrenamiento
from .configuracion import DIR_S2_MANO, VALOR_SIN_DATO
from .conjunto_datos import ConjuntoInundaciones, semilla_por_worker
from .lectura import leer_particion, leer_tif
from .modelo_unet import UNet

# índice (0-based) dentro de los 13 bandas de S2Hand: B1 B2 B3 B4 B5 B6 B7 B8 B8A B9 B10 B11 B12
BANDAS_S2 = {"B2": 1, "B3": 2, "B4": 3, "B8": 7, "B11": 11, "B12": 12}
CANALES_POR_MODO = {"sar": 2, "optico": len(BANDAS_S2), "sar_optico": 2 + len(BANDAS_S2)}
MODOS = list(CANALES_POR_MODO)


# ----------------------------------------------------------------------------- lectura S2
def leer_s2(id_chip: str, carpeta=DIR_S2_MANO) -> np.ndarray:
    """(6, H, W) float32 con las bandas elegidas, en las unidades del archivo."""
    s2 = leer_tif(carpeta / f"{id_chip}_S2Hand.tif").astype(np.float32)
    return s2[list(BANDAS_S2.values())]


def validos_s2(s2: np.ndarray) -> np.ndarray:
    """Píxel S2 válido: todas las bandas finitas y no todo cero (0 = relleno fuera de escena)."""
    return np.isfinite(s2).all(axis=0) & (np.nan_to_num(s2).sum(axis=0) > 0)


def calcular_estadisticas_s2(ids_train, n_por_chip: int = 5000, semilla: int = 42, ruta_json=None) -> dict:
    """p1, p99, media y desviación por banda, SOLO con train y con píxeles válidos (como en la fase 1)."""
    generador = np.random.default_rng(semilla)
    muestras = []
    for id_chip in tqdm(list(ids_train), desc="Estadísticas S2 (train)"):
        s2 = leer_s2(id_chip)
        valores = s2[:, validos_s2(s2)]
        if valores.shape[1] == 0:
            continue
        elegidos = generador.choice(valores.shape[1], min(n_por_chip, valores.shape[1]), replace=False)
        muestras.append(valores[:, elegidos])
    muestras = np.concatenate(muestras, axis=1)
    estadisticas = {}
    for i, banda in enumerate(BANDAS_S2):
        p1, p99 = np.percentile(muestras[i], [1, 99])
        recortado = np.clip(muestras[i], p1, p99)
        estadisticas[banda] = {"p1": float(p1), "p99": float(p99),
                               "media": float(recortado.mean()), "desviacion": float(recortado.std())}
    if ruta_json:
        with open(ruta_json, "w") as f:
            json.dump(estadisticas, f, indent=2)
    return estadisticas


class NormalizadorS2:
    """Mismo esquema que el del SAR: recorte a [p1, p99] -> estandarizar -> inválidos a 0."""

    def __init__(self, estadisticas: dict):
        a = lambda clave: np.array([estadisticas[b][clave] for b in BANDAS_S2], dtype=np.float32)[:, None, None]
        self.minimos, self.maximos, self.medias, self.desviaciones = a("p1"), a("p99"), a("media"), a("desviacion")

    @classmethod
    def desde_json(cls, ruta):
        with open(ruta) as f:
            return cls(json.load(f))

    def __call__(self, s2: np.ndarray):
        validos = validos_s2(s2)
        x = (np.clip(s2, self.minimos, self.maximos) - self.medias) / self.desviaciones
        x = np.nan_to_num(x, nan=0.0, posinf=0.0, neginf=0.0)
        x[:, ~validos] = 0.0
        return x.astype(np.float32), validos


# ----------------------------------------------------------------------------- dataset
class ConjuntoFusion(ConjuntoInundaciones):
    """Como ConjuntoInundaciones, pero arma la entrada según el modo y aplica la máscara común.
    La caché se guarda en float16 (8 canales x 446 chips no caben cómodos en float32 con 2 procesos)."""

    def __init__(self, ids_chips, normalizador, normalizador_s2, modo: str, **parametros):
        assert modo in CANALES_POR_MODO, f"modo desconocido: {modo}"
        self.normalizador_s2, self.modo = normalizador_s2, modo
        super().__init__(ids_chips, normalizador, **parametros)

    def _cargar(self, id_chip):
        sar, etiqueta = super()._cargar(id_chip)              # ya marca -1 donde el SAR no es finito
        optico, validos = self.normalizador_s2(leer_s2(id_chip))
        etiqueta[~validos] = VALOR_SIN_DATO                    # máscara común a los 3 modos
        imagen = {"sar": sar, "optico": optico, "sar_optico": np.concatenate([sar, optico])}[self.modo]
        return imagen.astype(np.float16), etiqueta

    def __getitem__(self, indice):
        imagen, etiqueta, id_chip = super().__getitem__(indice)
        return imagen.float(), etiqueta, id_chip


def crear_cargadores_fusion(modo, normalizador, normalizador_s2, tamano_lote: int = 8,
                            tamano_recorte: int = 256, num_workers: int = 2, semilla: int = 42,
                            particiones=("train", "valid", "test", "bolivia")) -> dict:
    cargadores = {}
    for particion in particiones:
        es_train = particion == "train"
        conjunto = ConjuntoFusion(leer_particion(particion)["id_chip"], normalizador, normalizador_s2, modo,
                                  aumentar_datos=es_train, tamano_recorte=tamano_recorte if es_train else None,
                                  semilla=semilla)
        cargadores[particion] = DataLoader(
            conjunto, batch_size=tamano_lote if es_train else max(1, tamano_lote // 2), shuffle=es_train,
            drop_last=es_train, num_workers=num_workers, pin_memory=torch.cuda.is_available(),
            worker_init_fn=semilla_por_worker, generator=torch.Generator().manual_seed(semilla),
            persistent_workers=num_workers > 0)
    return cargadores


# ----------------------------------------------------------------------------- modelo
@contextmanager
def canales_de_entrada(n_canales: int):
    """entrenamiento.crear_modelo fija 2 canales. Mientras dure el bloque, crea la U-Net con
    n_canales (sirve para entrenar() y cargar_mejor_modelo()) sin modificar entrenamiento.py."""
    original = _entrenamiento.crear_modelo

    def crear(configuracion):
        return UNet(canales_entrada=n_canales, canales_base=configuracion.canales_base,
                    profundidad=configuracion.profundidad, usar_saltos=configuracion.usar_saltos,
                    abandono=configuracion.abandono)

    _entrenamiento.crear_modelo = crear
    try:
        yield
    finally:
        _entrenamiento.crear_modelo = original


def cargar_modelo_fusion(dir_corrida, dispositivo):
    with open(dir_corrida / "fusion.json") as f:
        modo = json.load(f)["modo"]
    with canales_de_entrada(CANALES_POR_MODO[modo]):
        return _entrenamiento.cargar_mejor_modelo(dir_corrida, dispositivo)


# ----------------------------------------------------------------------------- diagnóstico
def diagnostico_s2(particiones=("train", "valid", "test", "bolivia"), umbral_brillo: float = 0.25):
    """Por partición: % de píxeles etiquetados (0/1) con S2 inválido y % 'brillantes' en el visible
    (B2 > umbral en reflectancia -> probable nube o bruma que quedó dentro de la etiqueta)."""
    from .lectura import leer_etiqueta
    import pandas as pd
    filas = []
    for particion in particiones:
        etiquetados = invalidos = brillantes = 0
        for id_chip in leer_particion(particion)["id_chip"]:
            s2, etiqueta = leer_s2(id_chip), leer_etiqueta(id_chip)
            validos = validos_s2(s2)
            escala = 10000.0 if validos.any() and np.median(s2[0][validos]) > 1.5 else 1.0   # DN x10000 o reflectancia
            con_etiqueta = etiqueta != VALOR_SIN_DATO
            etiquetados += con_etiqueta.sum()
            invalidos += (con_etiqueta & ~validos).sum()
            brillantes += (con_etiqueta & validos & (s2[0] / escala > umbral_brillo)).sum()
        filas.append({"particion": particion, "px_etiquetados": int(etiquetados),
                      "% S2 inválido": round(100 * invalidos / max(etiquetados, 1), 2),
                      "% visible brillante (posible nube)": round(100 * brillantes / max(etiquetados, 1), 2)})
    return pd.DataFrame(filas).set_index("particion")


In [ ]:
%%writefile /content/drive/MyDrive/rnap_p2_inundaciones/src/tarea_e6.py
"""
tarea_e6.py
Proceso TRABAJADOR del entregable 6. Se lanza uno por GPU (igual que en la fase 6a):
    CUDA_VISIBLE_DEVICES=1 python -m src.tarea_e6 --plan plan_e6.json --gpu 1
Cada tarea es "modo:semilla". Entrena (si falta), elige el mejor modelo por IoU de VALIDACIÓN
y lo evalúa en valid, test y Bolivia (umbral 0.5 fijo). Guarda todo en la carpeta de la corrida.
"""
import argparse
import gc
import json
from dataclasses import replace

import torch

from .configuracion import DIR_RESULTADOS
from .entrenamiento import entrenar
from .evaluacion import evaluar
from .experimento import ConfiguracionExperimento
from .fusion_optica import (CANALES_POR_MODO, NormalizadorS2, canales_de_entrada, cargar_modelo_fusion,
                            crear_cargadores_fusion)
from .preprocesamiento import Normalizador
from .utilidades import fijar_semilla, obtener_dispositivo

PARTICIONES_EVALUACION = ["valid", "test", "bolivia"]


def correr_tarea(modo, semilla, base, parametros, normalizador, normalizador_s2, dispositivo, dir_experimentos):
    configuracion = replace(base, nombre=f"e6_{modo}_s{semilla}", semilla=semilla)
    dir_corrida = dir_experimentos / configuracion.nombre
    dir_corrida.mkdir(parents=True, exist_ok=True)
    with open(dir_corrida / "fusion.json", "w") as f:
        json.dump({"modo": modo, "canales": CANALES_POR_MODO[modo]}, f)

    fijar_semilla(semilla)
    cargadores = crear_cargadores_fusion(modo, normalizador, normalizador_s2, parametros["tamano_lote"],
                                         parametros["tamano_recorte"], parametros["num_workers"], semilla)
    if not (dir_corrida / "resumen.json").exists():
        with canales_de_entrada(CANALES_POR_MODO[modo]):
            entrenar(configuracion, cargadores["train"], cargadores["valid"], dispositivo, dir_corrida)
        (dir_corrida / "ultimo.pt").unlink(missing_ok=True)
    else:
        print("  ya estaba terminada: se reutiliza", flush=True)

    modelo, _, _ = cargar_modelo_fusion(dir_corrida, dispositivo)
    for particion in PARTICIONES_EVALUACION:
        acumulador, _ = evaluar(modelo, cargadores[particion], dispositivo)
        with open(dir_corrida / f"metricas_{particion}.json", "w") as f:
            json.dump(acumulador.metricas(), f, indent=2)
        acumulador.tabla_por_evento().to_csv(dir_corrida / f"por_evento_{particion}.csv")
        print(f"  {particion:8s} IoU {acumulador.metricas()['iou']:.4f}", flush=True)
    del modelo, cargadores
    gc.collect(); torch.cuda.empty_cache()


def main():
    argumentos = argparse.ArgumentParser()
    argumentos.add_argument("--plan", required=True)
    argumentos.add_argument("--gpu", type=int, required=True)
    args = argumentos.parse_args()
    with open(args.plan) as f:
        plan = json.load(f)
    tareas = plan["tareas_por_gpu"][str(args.gpu)]
    print(f"[GPU {args.gpu}] tareas: {tareas}", flush=True)

    base = ConfiguracionExperimento(**plan["configuracion_base"])
    dispositivo = obtener_dispositivo()
    normalizador = Normalizador.desde_json(DIR_RESULTADOS / "f1_estadisticas_normalizacion.json")
    normalizador_s2 = NormalizadorS2.desde_json(DIR_RESULTADOS / "f10_estadisticas_s2.json")
    for tarea in tareas:
        modo, semilla = tarea.split(":")
        print(f"\n[GPU {args.gpu}] >>> {modo} semilla {semilla}", flush=True)
        correr_tarea(modo, int(semilla), base, plan["parametros"], normalizador, normalizador_s2, dispositivo,
                     DIR_RESULTADOS / "experimentos")
    print(f"\n[GPU {args.gpu}] TERMINADO", flush=True)


if __name__ == "__main__":
    main()


In [ ]:
%%writefile /content/drive/MyDrive/rnap_p2_inundaciones/src/paralelo_e6.py
"""
paralelo_e6.py
Reparte las 9 corridas del entregable 6 (3 modos x 3 semillas) entre las GPUs: un proceso por GPU,
igual que en la fase 6a. Reutiliza esperar() de paralelo_6a.py.
"""
import json
import os
import subprocess
import sys

import pandas as pd

from .fusion_optica import MODOS
from .paralelo_6a import esperar  # noqa: F401  (se reexporta para el notebook)

# minutos estimados en una T4 (solo para balancear; más canales = algo más de E/S)
DURACION_ESTIMADA = {"sar": 5, "optico": 6, "sar_optico": 6}


def armar_plan(semillas, n_gpus, configuracion_base, parametros, ruta_plan, modos=MODOS):
    tareas = [f"{m}:{s}" for m in modos for s in semillas]
    tareas.sort(key=lambda t: DURACION_ESTIMADA[t.split(":")[0]], reverse=True)
    cargas, asignacion = [0] * n_gpus, {str(g): [] for g in range(n_gpus)}
    for tarea in tareas:                                   # LPT: la más larga a la GPU menos cargada
        gpu = cargas.index(min(cargas))
        asignacion[str(gpu)].append(tarea)
        cargas[gpu] += DURACION_ESTIMADA[tarea.split(":")[0]]
    plan = {"tareas_por_gpu": asignacion, "configuracion_base": configuracion_base.__dict__, "parametros": parametros}
    with open(ruta_plan, "w") as f:
        json.dump(plan, f, indent=2)
    for gpu, lista in asignacion.items():
        print(f"GPU {gpu}: {lista}  (~{cargas[int(gpu)]} min estimados)")
    return plan


def lanzar(ruta_plan, n_gpus, raiz_proyecto, dir_logs):
    dir_logs.mkdir(parents=True, exist_ok=True)
    procesos = []
    for gpu in range(n_gpus):
        entorno = dict(os.environ, CUDA_VISIBLE_DEVICES=str(gpu), PYTHONUNBUFFERED="1")
        log = open(dir_logs / f"gpu{gpu}.log", "w")
        proceso = subprocess.Popen([sys.executable, "-m", "src.tarea_e6", "--plan", str(ruta_plan), "--gpu", str(gpu)],
                                   cwd=str(raiz_proyecto), env=entorno, stdout=log, stderr=subprocess.STDOUT)
        procesos.append((gpu, proceso, log))
        print(f"Lanzado proceso GPU {gpu} (pid {proceso.pid}) -> {dir_logs / f'gpu{gpu}.log'}")
    return procesos


def recolectar(semillas, dir_experimentos, modos=MODOS, particiones=("valid", "test", "bolivia")) -> pd.DataFrame:
    filas = []
    for modo in modos:
        for semilla in semillas:
            dir_corrida = dir_experimentos / f"e6_{modo}_s{semilla}"
            with open(dir_corrida / "resumen.json") as f:
                mejor_epoca = json.load(f)["mejor_epoca"]
            for particion in particiones:
                with open(dir_corrida / f"metricas_{particion}.json") as f:
                    filas.append({"modo": modo, "semilla": semilla, "particion": particion,
                                  "mejor_epoca": mejor_epoca, **json.load(f)})
    return pd.DataFrame(filas)


def resumen_por_modo(df, metricas=("iou", "f1", "precision", "recall", "omision", "comision")) -> pd.DataFrame:
    """Media ± desviación sobre semillas, por partición y modo."""
    agrupado = df.groupby(["particion", "modo"], sort=False)[list(metricas)]
    media, desviacion = agrupado.mean(), agrupado.std(ddof=1)
    return media.round(3).astype(str) + " ± " + desviacion.round(3).astype(str)


def deltas_pareados(df, referencia="sar", metrica="iou") -> pd.DataFrame:
    """Cambio de la métrica respecto a la referencia, emparejando por semilla (misma semilla = mismo
    orden de lotes y aumentos). Reporta media, desviación y si las 3 diferencias tienen el mismo signo."""
    tabla = df.pivot_table(index=["particion", "semilla"], columns="modo", values=metrica)
    filas = []
    for modo in [m for m in tabla.columns if m != referencia]:
        diferencias = (tabla[modo] - tabla[referencia]).groupby(level=0)
        for particion, d in diferencias:
            filas.append({"particion": particion, "comparacion": f"{modo} − {referencia}",
                          f"Δ{metrica} media": round(d.mean(), 4), f"Δ{metrica} desv.": round(d.std(ddof=1), 4),
                          "mismo signo en todas las semillas": bool((d > 0).all() or (d < 0).all())})
    return pd.DataFrame(filas)


def por_evento(dir_experimentos, semillas, particion="test", modos=MODOS) -> pd.DataFrame:
    columnas = {}
    for modo in modos:
        tablas = [pd.read_csv(dir_experimentos / f"e6_{modo}_s{s}" / f"por_evento_{particion}.csv", index_col=0)["iou"]
                  for s in semillas]
        columnas[modo] = pd.concat(tablas, axis=1).mean(axis=1)
    tabla = pd.DataFrame(columnas)
    tabla["Δ fusión − sar"] = tabla["sar_optico"] - tabla["sar"]
    return tabla.sort_values("Δ fusión − sar", ascending=False)


In [ ]:
%%writefile /content/drive/MyDrive/rnap_p2_inundaciones/src/comparacion_fusion.py
"""
comparacion_fusion.py
Predicción chip a chip para comparar SAR vs. fusión en casos concretos (los del entregable 5 y el
chip de test donde más gana la fusión).
"""
import numpy as np
import torch

from .evaluacion import predecir_probabilidades
from .fusion_optica import BANDAS_S2, ConjuntoFusion, cargar_modelo_fusion, leer_s2
from .lectura import leer_s1


def predecir_chips(ids, modo, semilla, normalizador, normalizador_s2, dir_experimentos, dispositivo) -> dict:
    """{id_chip: (probabilidad (H, W), etiqueta con máscara común (H, W))} para un modo y semilla."""
    modelo, _, _ = cargar_modelo_fusion(dir_experimentos / f"e6_{modo}_s{semilla}", dispositivo)
    conjunto = ConjuntoFusion(ids, normalizador, normalizador_s2, modo, en_memoria=False)
    salida = {}
    for i in range(len(conjunto)):
        imagen, etiqueta, id_chip = conjunto[i]
        salida[id_chip] = (predecir_probabilidades(modelo, imagen, dispositivo).numpy(), etiqueta.numpy())
    del modelo
    torch.cuda.empty_cache()
    return salida


def iou_chip(probabilidad, etiqueta, umbral=0.5) -> float:
    p = probabilidad > umbral
    vp = (p & (etiqueta == 1)).sum()
    fp = (p & (etiqueta == 0)).sum()
    fn = (~p & (etiqueta == 1)).sum()
    return float(vp / (vp + fp + fn)) if vp + fp + fn else float("nan")


def ejemplos_para_figura(ids, predicciones_por_modo) -> list:
    """Arma las tuplas que espera visualizacion_fusion.figura_comparacion."""
    indices_rgb = [list(BANDAS_S2).index(b) for b in ("B4", "B3", "B2")]
    ejemplos = []
    for id_chip in ids:
        vh = leer_s1(id_chip)[1]
        rgb = leer_s2(id_chip)[indices_rgb]
        etiqueta = predicciones_por_modo["sar"][id_chip][1]
        ejemplos.append((id_chip, np.nan_to_num(vh, nan=-30), rgb, etiqueta,
                         {m: p[id_chip][0] for m, p in predicciones_por_modo.items()}))
    return ejemplos


In [ ]:
%%writefile /content/drive/MyDrive/rnap_p2_inundaciones/src/visualizacion_fusion.py
"""
visualizacion_fusion.py
Figuras del entregable 6: barras de IoU por modo y comparación visual SAR vs. fusión en chips concretos.
"""
import matplotlib.pyplot as plt
import numpy as np

from .visualizacion_final import LEYENDA, MAPA_ERRORES, mapa_errores

COLORES = {"sar": "#1f5fbf", "optico": "#2ca02c", "sar_optico": "#9467bd"}
NOMBRES = {"sar": "SAR (VV, VH)", "optico": "Óptico (6 bandas S2)", "sar_optico": "Fusión SAR + óptico"}


def figura_barras(df, particiones=("valid", "test", "bolivia"), linea_otsu=None, ruta_guardado=None):
    """IoU medio ± desviación por modo y partición; los puntos son las semillas."""
    modos = list(dict.fromkeys(df["modo"]))
    figura, eje = plt.subplots(figsize=(7.5, 4))
    ancho = 0.8 / len(modos)
    for j, modo in enumerate(modos):
        for i, particion in enumerate(particiones):
            valores = df[(df["modo"] == modo) & (df["particion"] == particion)]["iou"].to_numpy()
            x = i + (j - (len(modos) - 1) / 2) * ancho
            eje.bar(x, valores.mean(), ancho * 0.9, yerr=valores.std(ddof=1) if len(valores) > 1 else 0,
                    color=COLORES.get(modo, "gray"), capsize=3, label=NOMBRES.get(modo, modo) if i == 0 else None)
            eje.scatter(np.full(len(valores), x), valores, color="black", s=8, zorder=3)
            eje.text(x, valores.mean() + 0.02, f"{valores.mean():.3f}", ha="center", fontsize=7)
    if linea_otsu is not None:
        for i, particion in enumerate(particiones):
            if particion in linea_otsu:
                eje.hlines(linea_otsu[particion], i - 0.45, i + 0.45, colors="gray", linestyles="--",
                           label="Otsu" if i == 0 else None)
    eje.set_xticks(range(len(particiones)), particiones)
    eje.set_ylabel("IoU agua"); eje.set_ylim(0, 1); eje.grid(axis="y", alpha=0.3)
    eje.set_title("Entregable 6 — SAR vs. óptico vs. fusión (3 semillas)")
    eje.legend(fontsize=8, loc="upper center", bbox_to_anchor=(0.5, -0.1), ncol=4, frameon=False)
    figura.tight_layout()
    if ruta_guardado:
        figura.savefig(ruta_guardado, dpi=150, bbox_inches="tight")
    return figura


def estirar_rgb(rgb, p_bajo=2, p_alto=98):
    """rgb: (3, H, W) en unidades del archivo -> (H, W, 3) en [0, 1] con estiramiento por percentiles."""
    validos = np.isfinite(rgb).all(axis=0) & (np.nan_to_num(rgb).sum(axis=0) > 0)
    salida = np.zeros(rgb.shape[1:] + (3,), dtype=np.float32)
    if validos.any():
        bajo, alto = np.percentile(rgb[:, validos], [p_bajo, p_alto])
        salida = np.clip((np.moveaxis(rgb, 0, -1) - bajo) / max(alto - bajo, 1e-6), 0, 1)
    return np.nan_to_num(salida)


def figura_comparacion(ejemplos, ruta_guardado=None):
    """ejemplos: [(id_chip, vh_db, rgb, etiqueta, {modo: probabilidad}), ...]
    -> VH | S2 RGB | errores SAR | errores fusión (con IoU del chip en el título)."""
    figura, ejes = plt.subplots(len(ejemplos), 4, figsize=(13, 3.4 * len(ejemplos)))
    ejes = np.atleast_2d(ejes)
    for fila, (id_chip, vh, rgb, etiqueta, probabilidades) in enumerate(ejemplos):
        ejes[fila, 0].imshow(vh, cmap="gray", vmin=-28, vmax=-8); ejes[fila, 0].set_title(f"{id_chip} — VH (dB)", fontsize=9)
        ejes[fila, 1].imshow(estirar_rgb(rgb)); ejes[fila, 1].set_title("Sentinel-2 color real", fontsize=9)
        for columna, modo in [(2, "sar"), (3, "sar_optico")]:
            p = probabilidades[modo] > 0.5
            validos = etiqueta != -1
            vp = (p & (etiqueta == 1)).sum(); fp = (p & (etiqueta == 0)).sum(); fn = (~p & (etiqueta == 1) & validos).sum()
            iou = vp / max(vp + fp + fn, 1)
            ejes[fila, columna].imshow(mapa_errores(etiqueta, probabilidades[modo]), cmap=MAPA_ERRORES, vmin=0, vmax=4,
                                       interpolation="nearest")
            ejes[fila, columna].set_title(f"Errores — {NOMBRES[modo]} (IoU {iou:.2f})", fontsize=9)
        for eje in ejes[fila]:
            eje.axis("off")
    figura.legend(handles=LEYENDA, loc="lower center", ncol=4, frameon=False)
    figura.tight_layout(rect=(0, 0.03, 1, 1))
    if ruta_guardado:
        figura.savefig(ruta_guardado, dpi=150, bbox_inches="tight")
    return figura


`paralelo_e6.py` reutiliza `esperar()` de `paralelo_6a.py`. Ese módulo solo se escribió en el notebook de Kaggle de la fase 6a,
así que **si no está en tu Drive** se crea aquí (si ya existe, no se toca).

In [ ]:
ruta_6a = RAIZ_PROYECTO / 'src' / 'paralelo_6a.py'
if ruta_6a.exists():
    print('paralelo_6a.py ya existe: no se modifica')
else:
    ruta_6a.write_text('"""\nparalelo_6a.py\nReparte las corridas de la ablación (a) entre varias GPUs: UN PROCESO POR GPU, cada uno con su\npropia lista de tareas (paralelismo por experimentos). Las corridas son independientes, así que\nlos resultados son idénticos a correrlas en serie, pero el tiempo total se reduce.\n\nReparto: algoritmo voraz LPT (la tarea más larga a la GPU menos cargada), con duraciones estimadas.\n"""\nimport json\nimport os\nimport subprocess\nimport sys\nimport time\nfrom collections import deque\n\nimport pandas as pd\n\n# minutos estimados en una T4 (solo para balancear la carga, no tienen que ser exactos)\nDURACION_ESTIMADA = {"combinada": 5, "debil": 30}\n\n\ndef armar_plan(semillas, n_gpus, configuracion_base, parametros, ruta_plan):\n    tareas = [f"combinada:{s}" for s in semillas] + [f"debil:{s}" for s in semillas]\n    tareas.sort(key=lambda t: DURACION_ESTIMADA[t.split(":")[0]], reverse=True)\n    cargas, asignacion = [0] * n_gpus, {str(g): [] for g in range(n_gpus)}\n    for tarea in tareas:\n        gpu = cargas.index(min(cargas))\n        asignacion[str(gpu)].append(tarea)\n        cargas[gpu] += DURACION_ESTIMADA[tarea.split(":")[0]]\n    plan = {"tareas_por_gpu": asignacion, "configuracion_base": configuracion_base.__dict__, "parametros": parametros}\n    with open(ruta_plan, "w") as f:\n        json.dump(plan, f, indent=2)\n    for gpu, lista in asignacion.items():\n        print(f"GPU {gpu}: {lista}  (~{cargas[int(gpu)]} min estimados)")\n    return plan\n\n\ndef lanzar(ruta_plan, n_gpus, raiz_proyecto, dir_logs):\n    dir_logs.mkdir(parents=True, exist_ok=True)\n    procesos = []\n    for gpu in range(n_gpus):\n        entorno = dict(os.environ, CUDA_VISIBLE_DEVICES=str(gpu), PYTHONUNBUFFERED="1")\n        log = open(dir_logs / f"gpu{gpu}.log", "w")\n        proceso = subprocess.Popen([sys.executable, "-m", "src.tarea_6a", "--plan", str(ruta_plan), "--gpu", str(gpu)],\n                                   cwd=str(raiz_proyecto), env=entorno, stdout=log, stderr=subprocess.STDOUT)\n        procesos.append((gpu, proceso, log))\n        print(f"Lanzado proceso GPU {gpu} (pid {proceso.pid}) -> {dir_logs / f\'gpu{gpu}.log\'}")\n    return procesos\n\n\ndef _ultimas_lineas(ruta, n=2):\n    with open(ruta, errors="ignore") as f:\n        lineas = [l.strip() for l in f.read().replace("\\r", "\\n").split("\\n") if l.strip()]\n    return list(deque(lineas, maxlen=n))\n\n\ndef esperar(procesos, dir_logs, cada_segundos: int = 60):\n    """Bloquea hasta que terminen todos; cada minuto imprime la última línea de cada GPU."""\n    inicio = time.time()\n    while any(p.poll() is None for _, p, _ in procesos):\n        time.sleep(cada_segundos)\n        minutos = (time.time() - inicio) / 60\n        for gpu, proceso, _ in procesos:\n            estado = "corriendo" if proceso.poll() is None else f"terminó (código {proceso.returncode})"\n            ultima = _ultimas_lineas(dir_logs / f"gpu{gpu}.log", 1)\n            print(f"[{minutos:5.1f} min] GPU {gpu} {estado}: {ultima[-1] if ultima else \'\'}", flush=True)\n    for gpu, proceso, log in procesos:\n        log.close()\n        if proceso.returncode != 0:\n            print(f"\\n⚠ GPU {gpu} falló (código {proceso.returncode}). Últimas líneas del log:")\n            print("\\n".join(_ultimas_lineas(dir_logs / f"gpu{gpu}.log", 25)))\n    print(f"Total: {(time.time() - inicio) / 60:.1f} min")\n    return all(p.returncode == 0 for _, p, _ in procesos)\n\n\ndef recolectar(semillas, dir_experimentos) -> pd.DataFrame:\n    """Lee de disco las métricas de validación de cada corrida (las escriben los trabajadores)."""\n    carpetas = {"solo_dorados": "combinada", "solo_debiles": "preentreno",\n                "mezcla": "mezcla", "preentreno_ajuste": "preentreno_ajuste"}\n    filas = []\n    for variante, carpeta in carpetas.items():\n        for semilla in semillas:\n            dir_corrida = dir_experimentos / f"{carpeta}_s{semilla}"\n            with open(dir_corrida / "metricas_valid.json") as f:\n                metricas = json.load(f)\n            with open(dir_corrida / "resumen.json") as f:\n                mejor_epoca = json.load(f)["mejor_epoca"]\n            filas.append({"variante": variante, "semilla": semilla, "mejor_epoca": mejor_epoca, **metricas})\n    return pd.DataFrame(filas)\n')
    print('paralelo_6a.py creado')

Recordatorio: si más adelante cambias algún módulo y vuelves a correr su `%%writefile`, **reinicia la sesión** antes de importarlo.

## 2. Imports y descarga (dorados + S2Hand)

In [ ]:
import json
import numpy as np
import pandas as pd
import torch
from src.configuracion import *
from src.descarga import descargar_mano, descargar_s2_mano
from src.preprocesamiento import Normalizador
from src.experimento import ConfiguracionExperimento
from src.lectura import leer_particion, leer_tif
from src.utilidades import obtener_dispositivo
from src.fusion_optica import BANDAS_S2, NormalizadorS2, calcular_estadisticas_s2, diagnostico_s2
from src.tarea_e6 import correr_tarea
from src.paralelo_e6 import recolectar, resumen_por_modo, deltas_pareados, por_evento
from src.visualizacion_fusion import figura_barras, figura_comparacion

dispositivo = obtener_dispositivo()
assert dispositivo.type == 'cuda', 'Esta fase necesita GPU: Entorno de ejecución > Cambiar tipo > T4 GPU'
descargar_mano()
descargar_s2_mano()
normalizador = Normalizador.desde_json(DIR_RESULTADOS / 'f1_estadisticas_normalizacion.json')
linea_base_otsu = pd.read_csv(DIR_RESULTADOS / 'f4_linea_base_otsu.csv', index_col=0)
LINEA_OTSU = linea_base_otsu['iou'].to_dict()

## 3. Inspección del S2 y normalización (solo train)
Primero confirmamos el formato real del archivo (13 bandas, tipo de dato, escala) antes de confiar en los índices de banda.

In [ ]:
import numpy as np
ids_train = leer_particion('train')['id_chip']
ejemplo = leer_tif(DIR_S2_MANO / f"{ids_train.iloc[0]}_S2Hand.tif")
print('forma:', ejemplo.shape, '| tipo:', ejemplo.dtype)
assert ejemplo.shape[0] == 13, 'Se esperaban 13 bandas en S2Hand'
print('mediana por banda (las 13):', [round(float(v), 3) for v in np.nanmedian(ejemplo.reshape(13, -1).astype(float), axis=1)])

estadisticas_s2 = calcular_estadisticas_s2(ids_train, ruta_json=DIR_RESULTADOS / 'f10_estadisticas_s2.json')
pd.DataFrame(estadisticas_s2).round(3)

**Diagnóstico de nubes:** % de píxeles CON etiqueta (0/1) cuyo S2 es inválido, y % con el azul muy brillante
(B2 > 0.25 de reflectancia → probable nube/bruma que quedó dentro de la etiqueta). Si es alto, el óptico tendrá ruido ahí.

In [ ]:
tabla_s2 = diagnostico_s2()
tabla_s2.to_csv(DIR_RESULTADOS / 'f10_diagnostico_s2.csv')
tabla_s2

## 4. Corridas en serie (3 modos × 3 semillas)
Imprime una línea por época. Si la estadística S2 ya existía en Drive, se reutiliza la misma (no se recalcula entre sesiones).

In [ ]:
base = ConfiguracionExperimento(perdida='combinada', epocas=60, paciencia=15)
SEMILLAS = [42, 7, 123]
MODOS = ['sar', 'optico', 'sar_optico']
PARAMETROS = dict(tamano_lote=8, tamano_recorte=256, num_workers=2)
DIR_EXPERIMENTOS = DIR_RESULTADOS / 'experimentos'
normalizador_s2 = NormalizadorS2.desde_json(DIR_RESULTADOS / 'f10_estadisticas_s2.json')

for modo in MODOS:
    for semilla in SEMILLAS:
        print(f"\n{'=' * 70}\n▶ {modo} semilla {semilla}\n{'=' * 70}")
        correr_tarea(modo, semilla, base, PARAMETROS, normalizador, normalizador_s2, dispositivo, DIR_EXPERIMENTOS)

## 5. Resultados: media ± desviación (3 semillas) por partición

In [ ]:
df_e6 = recolectar(SEMILLAS, DIR_EXPERIMENTOS)
df_e6.to_csv(DIR_RESULTADOS / 'f10_corridas.csv', index=False)
tabla_e6 = resumen_por_modo(df_e6)
tabla_e6.to_csv(DIR_RESULTADOS / 'f10_resumen.csv')
tabla_e6

## 6. Cambio de IoU respecto al SAR (pareado por semilla) — **la cifra del entregable 6**
Si la desviación es menor que la media y las 3 semillas tienen el mismo signo, el cambio es consistente.

In [ ]:
tabla_deltas = deltas_pareados(df_e6)
tabla_deltas.to_csv(DIR_RESULTADOS / 'f10_deltas.csv', index=False)
display(tabla_deltas)
figura_barras(df_e6, linea_otsu=LINEA_OTSU, ruta_guardado=DIR_FIGURAS / 'f10_barras_iou.png');

## 7. ¿Dónde ayuda la fusión? IoU por evento en test
Esperado según la fase 9: más ganancia en eventos con agua no especular o suelo árido oscuro (Pakistán, Somalia).

In [ ]:
tabla_eventos = por_evento(DIR_EXPERIMENTOS, SEMILLAS, 'test')
tabla_eventos.to_csv(DIR_RESULTADOS / 'f10_por_evento_test.csv')
tabla_eventos.round(3)

## 8. Comparación visual: casos del entregable 5 + chip de test donde más gana la fusión
Semilla 42. Columnas: VH, S2 en color real, errores del SAR y errores de la fusión.

In [ ]:
from src.comparacion_fusion import predecir_chips, iou_chip, ejemplos_para_figura
from src.lectura import leer_todas_las_particiones
particion_de = leer_todas_las_particiones().set_index('id_chip')['particion']

ids_test = leer_particion('test')['id_chip'].tolist()
CASOS_E5 = ['Pakistan_849790', 'Pakistan_70625', 'Bolivia_314919']
ids_figura = [i for i in CASOS_E5 if particion_de.get(i) in ('valid', 'test', 'bolivia')]
ids_todos = list(dict.fromkeys(ids_test + ids_figura))
predicciones = {m: predecir_chips(ids_todos, m, 42, normalizador, normalizador_s2, DIR_EXPERIMENTOS, dispositivo)
                for m in ['sar', 'sar_optico']}

ganancia = {i: iou_chip(*predicciones['sar_optico'][i]) - iou_chip(*predicciones['sar'][i]) for i in ids_test}
ganancia = pd.Series(ganancia).dropna().sort_values(ascending=False)
print('Chips de test con más ganancia de la fusión (IoU):'); display(ganancia.head(5).round(3))
print('Chips de test con MENOR ganancia (negativo = la fusión empeora):'); display(ganancia.tail(3).round(3))
ids_figura += [i for i in ganancia.index if i not in ids_figura][:1]   # el de más ganancia fuera de los casos E5
print('En la figura:', [(i, particion_de[i]) for i in ids_figura])
figura_comparacion(ejemplos_para_figura(ids_figura, predicciones), DIR_FIGURAS / 'f10_comparacion_casos.png');

## Para traer a la conversación
Salidas de las secciones **3** (forma/tipo + tabla de nubes), **5**, **6** (deltas + figura), **7** y **8** (ganancias + figura).